# CIDS · 04 — S3a: SegResNet → SAM, fully automatic (first council step)

No training and **no ground truth used for prompting**:
1. SegResNet (image-only, Exp 1 `best.pt`) predicts the tumour at 1 mm.
2. Each predicted region (connected component ≥ 0.1 ml, top 5) becomes a candidate.
3. For every candidate: a 128³ crop at 1.5 mm centred on it, and its bounding box as the prompt → SAM-Med3D-turbo encoder → our fine-tuned decoder (S2a LoRA `best.pt`).
4. Jury: agreement = Dice(SAM mask, SegResNet candidate).

Rules compared (chosen on the 120 validation cases, applied once to the 306 test cases): `seg_rel20` (S1 baseline, 0.655), `sam_rel20`, `avg_rel20`, `jury_0.3/0.5/0.7`.

**Settings:** GPU T4 x2 · **Internet ON** · Persistence: Files.  
**Inputs:** the 2-channel cache **and** `cids-checkpoints` with two files: `segresnet_img_best.pt` and `sam_lora_best.pt`.  
**Run:** smoke test first, then *Save Version → Save & Run All* (~1.5–2 h).

In [ ]:
REPO_URL            = 'https://github.com/Vemuri-BK/cids.git'
GIT_BRANCH          = 'main'
EXPECTED_GIT_COMMIT = ''      # paste the full commit id you pushed

CACHE_ROOT = ''               # '' = auto-detect the 2-channel cache
SEG_CKPT   = ''               # '' = auto-detect segresnet_img_best.pt under /kaggle/input
SAM_CKPT   = ''               # '' = auto-detect sam_lora_best.pt under /kaggle/input

CODE_DIR = '/kaggle/working/cids'
OUT      = '/kaggle/working/s3a_seg2sam'

In [ ]:
import os, subprocess
if os.path.isdir(f'{CODE_DIR}/.git'):
    subprocess.run(['git', '-C', CODE_DIR, 'fetch', 'origin', GIT_BRANCH], check=True)
    subprocess.run(['git', '-C', CODE_DIR, 'checkout', GIT_BRANCH], check=True)
    subprocess.run(['git', '-C', CODE_DIR, 'pull', 'origin', GIT_BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '-b', GIT_BRANCH, REPO_URL, CODE_DIR], check=True)
if EXPECTED_GIT_COMMIT:
    subprocess.run(['git', '-C', CODE_DIR, 'checkout', EXPECTED_GIT_COMMIT], check=True)
head = subprocess.check_output(['git', '-C', CODE_DIR, 'rev-parse', 'HEAD'], text=True).strip()
assert not EXPECTED_GIT_COMMIT or head == EXPECTED_GIT_COMMIT, (head, EXPECTED_GIT_COMMIT)
print('code at commit', head)
%cd {CODE_DIR}

In [ ]:
!pip install -q monai medim einops appdirs
!pytest -q tests

In [ ]:
import sys, torch, shutil
from pathlib import Path
assert torch.cuda.is_available(), 'No GPU! Settings -> Accelerator -> GPU T4 x2, then restart.'
print('GPU:', torch.cuda.get_device_properties(0).name)
sys.path.insert(0, CODE_DIR); sys.path.insert(0, f'{CODE_DIR}/scripts')
from cids.data.cache import find_cache_root
from s3a_seg2sam import find_file
CACHE_ROOT = str(find_cache_root(CACHE_ROOT))
SEG_CKPT = str(find_file('segresnet_img_best.pt', SEG_CKPT))
SAM_CKPT = str(find_file('sam_lora_best.pt', SAM_CKPT))
print('cache:', CACHE_ROOT); print('SegResNet:', SEG_CKPT); print('SAM LoRA:', SAM_CKPT)
if os.path.exists(OUT):                  # never mix with a previous run
    shutil.rmtree(OUT); print('removed old', OUT)

## Smoke test — 4 val + 4 test cases (~5 min)
Check: no errors; `seg_rel20` Dice is a normal number (not ~0); SAM rules give similar or better Dice. A candidate should lie on the tumour in most cases.

In [ ]:
!python scripts/s3a_seg2sam.py --cache_root {CACHE_ROOT} --seg_ckpt {SEG_CKPT} --sam_lora_ckpt {SAM_CKPT} \
    --out /kaggle/working/s3a_smoke --limit_eval 4
!rm -rf /kaggle/working/s3a_smoke

## Full run — 120 val + 306 test (~1.5–2 h) — use *Save Version → Save & Run All*

In [ ]:
!python scripts/s3a_seg2sam.py --cache_root {CACHE_ROOT} --seg_ckpt {SEG_CKPT} --sam_lora_ckpt {SAM_CKPT} --out {OUT}

In [ ]:
import json, pandas as pd
print(json.dumps(json.load(open(f'{OUT}/summary.json')), indent=2))
te = pd.read_csv(f'{OUT}/test_metrics.csv')
print(te.pivot_table(index='collection', columns='rule', values='dice').round(3))
!du -sh {OUT}